In [1]:

import csv
import io


def find_s(examples, labels):
    """
    examples: list of rows, each row is a list of attribute values
    labels:   list of 'Yes'/'No' (or 1/0) target labels, same length as examples
    Returns: final hypothesis (list), and a log of hypothesis after each
             positive example (for teaching/inspection purposes).
    """
    hypothesis = None
    history = []

    for i, (row, label) in enumerate(zip(examples, labels)):
        is_positive = str(label).strip().lower() in ("yes", "1", "true")

        if not is_positive:
            # Find-S completely IGNORES negative examples.
            continue

        if hypothesis is None:
            # First positive example becomes the initial hypothesis exactly.
            hypothesis = list(row)
        else:
            # Generalize any attribute that doesn't match the current example.
            for j in range(len(hypothesis)):
                if hypothesis[j] != row[j]:
                    hypothesis[j] = '?'

        history.append((i, list(hypothesis)))

    return hypothesis, history


def read_dataset_from_user():
    """
    Reads a small CSV-style dataset typed by the user, line by line.
    Format: comma-separated attribute values, LAST column is the label
    (Yes/No). First line must be the header.
    Type 'done' on an empty prompt to use the built-in demo dataset instead.
    """
    print("Enter your dataset as CSV rows (last column = label Yes/No).")
    print("First line = header. Type 'done' when finished, or press Enter")
    print("immediately to use the built-in EnjoySport demo dataset.\n")

    lines = []
    first = input("Header row (or Enter for demo): ").strip()
    if first == "":
        return demo_dataset()

    lines.append(first)
    while True:
        line = input("Row (or 'done'): ").strip()
        if line.lower() == "done":
            break
        lines.append(line)

    reader = csv.reader(io.StringIO("\n".join(lines)))
    rows = list(reader)
    header, data_rows = rows[0], rows[1:]
    examples = [r[:-1] for r in data_rows]
    labels = [r[-1] for r in data_rows]
    return header[:-1], examples, labels


def demo_dataset():
    """Classic EnjoySport dataset from Tom Mitchell's textbook."""
    header = ["Sky", "AirTemp", "Humidity", "Wind", "Water", "Forecast"]
    examples = [
        ["Sunny", "Warm", "Normal", "Strong", "Warm", "Same"],
        ["Sunny", "Warm", "High",   "Strong", "Warm", "Same"],
        ["Rainy", "Cold", "High",   "Strong", "Warm", "Change"],
        ["Sunny", "Warm", "High",   "Strong", "Cool", "Change"],
    ]
    labels = ["Yes", "Yes", "No", "Yes"]
    return header, examples, labels


if __name__ == "__main__":
    # For an interactive run, call read_dataset_from_user() instead.
    header, examples, labels = demo_dataset()

    print("Dataset:")
    print(f"{'  '.join(header):50s} Label")
    for row, label in zip(examples, labels):
        print(f"{'  '.join(row):50s} {label}")

    print("\nRunning Find-S...\n")
    final_hypothesis, history = find_s(examples, labels)

    print("Hypothesis after each POSITIVE example:")
    for idx, h in history:
        print(f"  After example {idx + 1} ({labels[idx]}): {h}")

    print(f"\nFinal most-specific hypothesis consistent with all positive examples:")
    print(f"  {final_hypothesis}")


Dataset:
Sky  AirTemp  Humidity  Wind  Water  Forecast      Label
Sunny  Warm  Normal  Strong  Warm  Same            Yes
Sunny  Warm  High  Strong  Warm  Same              Yes
Rainy  Cold  High  Strong  Warm  Change            No
Sunny  Warm  High  Strong  Cool  Change            Yes

Running Find-S...

Hypothesis after each POSITIVE example:
  After example 1 (Yes): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
  After example 2 (Yes): ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  After example 4 (Yes): ['Sunny', 'Warm', '?', 'Strong', '?', '?']

Final most-specific hypothesis consistent with all positive examples:
  ['Sunny', 'Warm', '?', 'Strong', '?', '?']
